In [1]:
import random
import pandas as pd
from datasets import Dataset
import os 
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from datasets import load_dataset
import seaborn as sns
from datasets import load_dataset, load_from_disk
import numpy as np
import json

In [5]:
cd /mnt/home/al2644/research/projects/perturb-r/

/mnt/home/al2644/research/projects/perturb-r


In [124]:
path="./results/math8k/benchmark/QwQ-32Btrain.pickle"
df = pd.read_pickle(path)

In [110]:
df['is_correct'] = df.apply(lambda x: 1.0 if x["score"] == 1.0 or x["math_score"] == 1.0 else 0., axis = 1)

In [128]:
df[df['is_correct']==1.0]

,problem,solution,level,subject,source,response,pred,gt,if_boxed,is_correct
0,Let $a$ and $b$ be the two real values of $x$ ...,118,5,Intermediate Algebra,math8k,"Alright, so I have this equation here: the cub...",Given the equation:\n\n\[\n\sqrt[3]{x} + \sqrt...,118,True,1.0
1,For how many integer values of $x$ is $5x^{2}+...,5,5,Algebra,math8k,"Okay, so I need to figure out for how many int...",To determine for how many integer values of \(...,5,True,1.0
2,A car is averaging 50 miles per hour. If the c...,30,3,Prealgebra,math8k,"Okay, so the problem is about a car traveling ...",To determine how many minutes less a 450-mile ...,30,True,1.0
3,Find the greatest common divisor of $10293$ an...,1,3,Number Theory,math8k,"Okay, so I need to find the greatest common di...",To find the greatest common divisor (GCD) of 1...,1,True,1.0
4,How many ounces of pure water must be added to...,15,3,Algebra,math8k,"Okay, so I need to figure out how many ounces ...",To determine how many ounces of pure water mus...,15,True,1.0
...,...,...,...,...,...,...,...,...,...,...
8516,"If three coins are tossed at the same time, wh...",\frac{3}{8},3,Counting & Probability,math8k,"Okay, so I need to find the probability of get...",To find the probability of getting exactly two...,\frac{3}{8},True,1.0
8517,If the odds for pulling a prize out of the box...,\frac{4}{7},3,Counting & Probability,math8k,"Okay, so I need to figure out the probability ...",The odds for pulling a prize out of the box ar...,\frac{4}{7},True,1.0
8519,"How many numbers in the list $43$, $4343$, $43...",1,3,Prealgebra,math8k,"Okay, so I need to figure out how many numbers...",To determine how many numbers in the list \(43...,1,True,1.0
8521,"A car travels 40 kph for 20 kilometers, 50 kph...",51,5,Prealgebra,math8k,"Okay, so I need to find the average speed of a...","To find the average speed of the car, we need ...",51,True,1.0


In [129]:
tokenizer = AutoTokenizer.from_pretrained("aochongoliverli/Qwen2.5-3B-Zero-Base")

In [130]:
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'Please reason step by step. Think through the problem in depth before answering. Finally, put your final answer within \\boxed{}.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
      